**Cell #01**

# RAG11 — Stage 2 (Yoga-Sūtra): Ask Smart Questions, Some in Devanagari

Runs five hand-picked questions about the **Yoga-Sūtra of Patañjali and the Yoga-Bhāṣya** through the same
retrieval + generation pipeline as the nutrition notebooks, but against the one source that holds them:
Michel Angot's French edition (`Yogasutra.janvier. 2020.pdf, éd. 2021.pdf`, chunked by
`stage1_1_eda_packages/source18_yoga_sutra_angot.py`).

What is different from the nutrition notebooks:

1. **One source only.** `ask_question(filter_owner=...)` restricts every retrieval step to this book's
   `rag11_data_sources.rowGUID`, so nutrition chunks can't compete.
2. **A Yoga-specific system prompt** (`ask_question(system_prompt=...)`) instead of the nutrition one.
3. **Devanagari questions.** The book stores Sanskrit as IAST transliteration inside French prose
   (`yogaś cittavṛttinirodhaḥ`), so a question typed as `योगश्चित्तवृत्तिनिरोधः` shares no characters with any chunk.
   `romanize_devanagari()` (in `reusable_code/devanagari.py`) rewrites every Devanagari run into IAST and the
   notebook appends that to the question: retrieval gets something to match, the model still sees the original.
4. **No parent expansion.** In this book a "parent" is one whole sūtra with its Bhāṣya and Angot's notes
   (often 10 000+ characters), and the excerpt would only show its beginning. The 400-token child chunks
   are the precise units here, so `expand_to_parents=False`.

**Before running this notebook**: `stage1_1` must have chunked this book with
`MAX_NUMBER_OF_PAGES_TO_USE = None` (with the default cap of 100 pages the sūtra text itself, which starts on
page 244, is empty) and `stage1_2` must have loaded it. Cell #05 checks this and tells you what is missing.

In [1]:
# Cell #02
from reusable_code import (
    init_clients, ask_question, contains_devanagari, romanize_devanagari, GENERATION_MODEL,
)
from reusable_code.env import optional_env

clients = init_clients()
print("Clients ready. Supabase project:", optional_env("PUBLIC_SUPABASE_URL"), "| model:", GENERATION_MODEL)

Clients ready. Supabase project: https://czgrxgzdmodkkmbmraub.supabase.co | model: claude-sonnet-5


**Cell #03**

## Find the book in the database and check it is fully loaded

Looks the source up by file name (its `source_key` can change if files are added to the Drive folder),
then counts its child chunks and its per-sūtra parent sections. The full book has 195 sūtra sections.

In [2]:
# Cell #04
ys_rows = (clients.supabase.table("rag11_data_sources").select('"rowGUID",source_key,filename')
           .ilike("filename", "%Yogasutra%").execute().data)
if not ys_rows:
    raise RuntimeError("The Yoga-Sutra book is not in rag11_data_sources -- run stage1_1 and stage1_2 first.")

YS_OWNER_GUID = ys_rows[0]["rowGUID"]

n_children = (clients.supabase.table("rag11_chunks_child_table").select('"rowGUID"', count="exact")
              .eq("rowOwnerGUID", YS_OWNER_GUID).limit(1).execute().count)
n_sutra_parents = (clients.supabase.table("rag11_chunks_parent_table").select('"rowGUID"', count="exact")
                   .eq("rowOwnerGUID", YS_OWNER_GUID).like("title", "Yoga-S%tra%").limit(1).execute().count)

print(f"{ys_rows[0]['source_key']}: {n_children} child chunks, {n_sutra_parents} of 195 sutra sections loaded")
if n_sutra_parents < 195:
    print("\nWARNING: the sutra text is not (fully) loaded, so questions about individual sutras will get "
          "'the excerpts do not contain...' answers.\n"
          "  1. stage1_1_extract_and_chunk.ipynb: set MAX_NUMBER_OF_PAGES_TO_USE = None and re-run\n"
          "  2. stage1_2_eda_load_chunks.ipynb: re-run (only new/changed chunks are embedded)\n"
          "  3. stage1_9_eda_verify_all_data.ipynb: confirm PASS")

source18: 314 child chunks, 0 of 195 sutra sections loaded

  1. stage1_1_extract_and_chunk.ipynb: set MAX_NUMBER_OF_PAGES_TO_USE = None and re-run
  2. stage1_2_eda_load_chunks.ipynb: re-run (only new/changed chunks are embedded)
  3. stage1_9_eda_verify_all_data.ipynb: confirm PASS


**Cell #05**

## The Yoga-specific system prompt

Same contract as the nutrition prompt (answer only from the numbered excerpts; a `Short answer: Yes|No` line
only when the question has a clean yes/no answer) plus what a reader of *this* book needs: who is speaking
(sūtra, Bhāṣya, or Angot), sūtra references, IAST for Sanskrit, and the answer language.

In [3]:
# Cell #06
YS_SYSTEM_PROMPT = """You are a research assistant for a French scholarly edition of the Yoga-Sutra of \
Patanjali and the Yoga-Bhasya of Vyasa (Michel Angot, 3rd edition 2021). Answer strictly using the numbered \
excerpts in the user message -- do not rely on outside knowledge, and say plainly if the excerpts don't \
contain enough information to answer.

The excerpts are mostly French; Sanskrit appears in IAST transliteration. The question may be in English, \
French, or Devanagari (Sanskrit or Hindi); a line "[IAST: ...]" after the question is its transliteration, \
added to help retrieval. Answer in the language of the question (English for a Sanskrit-only question).

Whenever the excerpts allow it: give the sutra reference (e.g. II.35), quote the key Sanskrit term in IAST, \
and say whether a claim comes from the Sutra itself, from the Bhasya, or from Angot's own commentary.

First decide whether the question has a clean Yes/No answer:
  - If it does, begin your reply with exactly this one line:
        Short answer: Yes
    or
        Short answer: No
    then a blank line, then the full explanation.
  - Otherwise (a "what"/"how"/"why" question), skip the "Short answer" line and give the full explanation.

Keep the explanation grounded in the excerpts."""


def prepare_question(question: str) -> str:
    """Append an IAST rendering of any Devanagari in the question (kept alongside the original)."""
    if not contains_devanagari(question):
        return question
    return f"{question}\n[IAST: {romanize_devanagari(question)}]"


def ask_ys(question: str, **overrides) -> dict:
    options = dict(
        match_count=6,
        use_hybrid=True,            # dense + keyword: exact sutra terms (IAST) are found by the keyword half
        use_multi_query=True,       # split compound questions into sub-questions
        use_hyde=False,
        use_rerank=True,            # cross-encoder re-scores the wide candidate pool
        expand_to_parents=False,    # a parent here is a whole sutra + commentary; keep the precise child chunks
        filter_owner=YS_OWNER_GUID,
        system_prompt=YS_SYSTEM_PROMPT,
    )
    options.update(overrides)
    return ask_question(prepare_question(question), **options)

**Cell #07**

## The five questions

Each targets a different part of the book and a different retrieval difficulty:

1. **Devanagari sūtra + a "how does the Bhāṣya explain" question**: the sūtra (I.2) is quoted in Devanagari, so
   only the IAST rendering can reach the chunk; the answer needs both the sūtra and the Bhāṣya's gloss of *nirodha*.
2. **Interpretive question about Angot's method** (English): the answer lives in one of his appendix notices
   (*Adhikāra I.1 et les destinataires du Yoga-Sūtra*), written in French, so this is cross-lingual retrieval.
3. **Devanagari sūtra + a consequence** (II.35): quote in Devanagari, ask what follows from it; the Bhāṣya and
   Angot's notes add nuance beyond the one-line sūtra.
4. **A multi-part comparison** (English, IAST terms): the five *vṛtti*-s and which are *kliṣṭa* / *akliṣṭa*
   (I.5-11): spread across seven sūtras, the case multi-query splitting exists for.
5. **A Hindi yes/no question written entirely in Devanagari**: *is Īśvara in the Yoga-Sūtra a creator god?* A
   clean Yes/No with an important nuance (I.23-26, II.1, II.45), and no Latin letters at all in the question.

In [4]:
# Cell #08
YS_QUESTIONS = [
    # 1. Devanagari sutra (I.2) + English question about the Bhasya's gloss of nirodha.
    "योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?",
    # 2. Interpretive: Angot's reading of the very first word, answer is in a French appendix notice.
    "Why does Angot read the opening word 'atha' of Yoga-Sūtra I.1 as an adhikāra, and who are the intended addressees of the text?",
    # 3. Devanagari sutra (II.35) + what follows from it.
    "अहिंसाप्रतिष्ठायां तत्सन्निधौ वैरत्यागः -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?",
    # 4. Multi-part comparison across I.5-I.11 -- the case for multi-query splitting.
    "How do the five vṛtti-s (pramāṇa, viparyaya, vikalpa, nidrā, smṛti) differ from one another, and which of them can be kliṣṭa or akliṣṭa?",
    # 5. Hindi, all Devanagari, clean Yes/No with a nuance.
    "क्या योगसूत्र में ईश्वर जगत् का सृष्टिकर्ता है?",
]

**Cell #09**

## What the Devanagari questions turn into

`romanize_devanagari()` only touches Devanagari characters; Latin text, digits and punctuation stay as they are.

In [5]:
# Cell #10
for i, q in enumerate(YS_QUESTIONS, start=1):
    if contains_devanagari(q):
        print(f"Q{i} original: {q}")
        print(f"Q{i} IAST    : {romanize_devanagari(q)}\n")

Q1 original: योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?
Q1 IAST    : yogaścittavṛttinirodhaḥ -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?

Q3 original: अहिंसाप्रतिष्ठायां तत्सन्निधौ वैरत्यागः -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?
Q3 IAST    : ahiṃsāpratiṣṭhāyāṃ tatsannidhau vairatyāgaḥ -- according to II.35, what happens in the presence of someone firmly established in ahiṃsā?

Q5 original: क्या योगसूत्र में ईश्वर जगत् का सृष्टिकर्ता है?
Q5 IAST    : kyā yogasūtra meṃ īśvara jagat kā sṛṣṭikartā hai?



**Cell #11**

## Run all 5 questions

Prints the pipeline's own bookkeeping too: which sub-questions multi-query produced, how many candidates were
considered before the reranker cut them to the final six excerpts, and the words the answer shares with them.

In [ ]:
# Cell #12
ys_results = []
for question in YS_QUESTIONS:
    print("Q:", question)
    result = ask_ys(question)
    ys_results.append(result)

    if result["subquestions"] and len(result["subquestions"]) > 1:
        print("  sub-questions:", " | ".join(result["subquestions"]))
    print(f"  candidates considered: {result['candidates_considered']} -> chunks used: {result['chunks_used']}")
    print(f"  source pages: {result['source_pages']}")
    if result["short_answer"]:
        print(f"  Short answer: {result['short_answer']}")
    print()
    print(result["answer"])
    print()
    grounding = ", ".join(result["grounding_words"]) or "(no shared terms found with the retrieved excerpts)"
    print(f"  established on: {grounding}")
    print("\n" + "-" * 80 + "\n")

Q: योगश्चित्तवृत्तिनिरोधः -- what does Yoga-Sūtra I.2 define yoga as, and how does the Bhāṣya explain the word nirodha?
  sub-questions: What does Yoga-Sūtra I.2 (yogaścittavṛttinirodhaḥ) define yoga as? | How does the Bhāṣya explain the word nirodha in Yoga-Sūtra I.2?
  candidates considered: 24 -> chunks used: 6
  source pages: [71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110]

Yoga-Sūtra I.2 (cittavṛttinirodhaḥ) presents what Angot proposes to translate as "Yoga is the stopping [nirodha] of the fluctuations [vṛtti] of the mental [citta]" (« Le yoga est l'arrêt des fluctuations du mental »). However, Angot notes that this sūtra's word order is unusual for a definition: in Pāṇinian style and in the rest of the Yoga-Sūtra, the defined term normally precedes the name being defined (one would expect *cittavṛttinirodho yogaḥ). Because of this, Angot suggests the sūtra 

**Cell #13**

## Summary table

In [ ]:
# Cell #14
print(f"{'#':<3} {'script':<11} {'short answer':<13} {'sub-Qs':>6} {'chunks':>7} {'pages':>6}  question")
for i, (q, r) in enumerate(zip(YS_QUESTIONS, ys_results), start=1):
    script = "Devanagari" if contains_devanagari(q) else "Latin"
    short = r["short_answer"] or "n/a"
    n_sub = len(r["subquestions"] or [])
    print(f"{i:<3} {script:<11} {short:<13} {n_sub:>6} {r['chunks_used']:>7} {len(r['source_pages']):>6}  {q[:70]}")

**Cell #15**

## Try it without the Devanagari help

To see what `romanize_devanagari()` buys, ask question 3 again with the transliteration switched off (the
model still gets the Devanagari, but retrieval only has the Devanagari to work with). Compare the pages and the answer.

In [ ]:
# Cell #16
raw = ask_question(
    YS_QUESTIONS[2], match_count=6, use_hybrid=True, use_multi_query=True, use_hyde=False, use_rerank=True,
    expand_to_parents=False, filter_owner=YS_OWNER_GUID, system_prompt=YS_SYSTEM_PROMPT,
)
with_iast = ys_results[2]
print("without IAST -> pages:", raw["source_pages"], "| chunks:", raw["chunks_used"])
print("with IAST    -> pages:", with_iast["source_pages"], "| chunks:", with_iast["chunks_used"])
print("\nanswer without IAST:\n" + raw["answer"][:600])

**Cell #17**

## Save workspace to GitHub

Synchronize this notebook and any code changes to GitHub (with auto lock recovery and conflict resolution).

In [ ]:
# Cell #18
from reusable_code import save_to_github

save_to_github("stage2_ask_examples7_ys.ipynb - Yoga-Sutra questions incl. Devanagari")